# Identifikasi Dataset Klasifikasi Multilabel

**Nama:** Mochamad Syauqil Huda  
**Kelas:** 05TPLP022  
**Mata Kuliah:** Machine Learning

## Tujuan
Notebook ini digunakan untuk mengidentifikasi tiga dataset yang termasuk klasifikasi **multilabel**. Pada klasifikasi multilabel, satu data dapat memiliki lebih dari satu label target secara bersamaan.

## Kriteria multilabel
Berbeda dengan multiclass yang hanya memiliki satu kelas target untuk setiap data, multilabel memiliki beberapa kolom target biner. Nilai `1` berarti label tersebut dimiliki oleh data, sedangkan `0` berarti label tersebut tidak dimiliki.


## 1. Persiapan Folder dan Dataset

Dataset yang dipilih berukuran relatif kecil dibandingkan dataset gambar atau teks berukuran besar. Ketiganya tersedia dalam format ARFF.

Buat struktur folder seperti berikut:

```text
PRAKTIKUM MACHINE LEARNING/
└── KLASIFIKASI MULTILABEL/
    ├── 03_Klasifikasi_Multilabel.ipynb
    └── dataset/
        ├── flags.arff
        ├── emotions.arff
        └── birds.arff
```

Unduh file `.arff` dari tautan yang dicantumkan pada setiap bagian dataset. Letakkan file tersebut langsung di dalam folder `dataset`.


In [ ]:
%pip install pandas matplotlib scipy

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 140)

# Lokasi kerja notebook di VS Code
folder_kerja = Path.cwd()

# Mendukung notebook dijalankan dari folder proyek utama
# atau langsung dari folder KLASIFIKASI MULTILABEL
kandidat_folder = [
    folder_kerja / "dataset",
    folder_kerja / "KLASIFIKASI MULTILABEL" / "dataset"
]

folder_dataset = next(
    (folder for folder in kandidat_folder if folder.is_dir()),
    None
)

if folder_dataset is None:
    raise FileNotFoundError(
        "Folder dataset tidak ditemukan. Buat folder 'dataset' "
        "di dalam folder KLASIFIKASI MULTILABEL."
    )

print("Library berhasil diimpor.")
print("Folder kerja:", folder_kerja.resolve())
print("Folder dataset:", folder_dataset.resolve())

## 2. Fungsi untuk Membaca dan Memeriksa Dataset Multilabel

File ARFF menyimpan nama atribut dan isi data. Pada ketiga dataset ini, kolom-kolom label berada di bagian akhir file. Jumlah kolom label ditentukan berdasarkan dokumentasi dataset.

Fungsi di bawah akan:
- membaca file ARFF;
- memisahkan fitur dan kolom label;
- menampilkan ukuran data dan contoh isi;
- menghitung jumlah label positif pada setiap kolom;
- memeriksa apakah ada data yang memiliki lebih dari satu label aktif.


In [ ]:
def baca_arff_multilabel(nama_file, jumlah_label):
    path_file = folder_dataset / nama_file

    if not path_file.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {path_file}\n"
            f"Letakkan file {nama_file} di dalam folder dataset."
        )

    data_arff, metadata = arff.loadarff(path_file)

    # Ubah data ARFF menjadi DataFrame
    df = pd.DataFrame(data_arff)
    df.columns = metadata.names()

    # Ubah nilai byte menjadi teks biasa
    for kolom in df.columns:
        df[kolom] = df[kolom].apply(
            lambda nilai: nilai.decode("utf-8") if isinstance(nilai, bytes) else nilai
        )

    if jumlah_label >= len(df.columns):
        raise ValueError("Jumlah label tidak valid untuk jumlah atribut dataset.")

    # Pada format dataset ini, kolom label berada di bagian paling akhir
    kolom_label = list(df.columns[-jumlah_label:])
    kolom_fitur = list(df.columns[:-jumlah_label])

    # Konversi label menjadi angka 0 dan 1
    for kolom in kolom_label:
        df[kolom] = df[kolom].astype(str).str.strip().str.lower().map({
            "1": 1, "1.0": 1, "true": 1, "yes": 1,
            "0": 0, "0.0": 0, "false": 0, "no": 0
        })

    if df[kolom_label].isnull().any().any():
        raise ValueError(
            "Ada nilai label yang tidak berhasil dikenali sebagai 0 atau 1. "
            "Periksa format file ARFF."
        )

    X = df[kolom_fitur]
    Y = df[kolom_label].astype(int)

    return df, X, Y


def periksa_multilabel(nama_dataset, nama_file, jumlah_label, url_sumber):
    print("=" * 80)
    print("NAMA DATASET:", nama_dataset)
    print("SUMBER:", url_sumber)
    print("=" * 80)

    df, X, Y = baca_arff_multilabel(nama_file, jumlah_label)

    print("\nUkuran dataset:", df.shape)
    print("Jumlah data:", len(df))
    print("Jumlah fitur:", X.shape[1])
    print("Jumlah kolom label:", Y.shape[1])

    print("\nContoh data:")
    display(df.head())

    print("\nJumlah missing value:")
    print(df.isnull().sum().to_frame("Jumlah Missing Value"))

    # Jumlah data yang memiliki label aktif pada setiap kolom
    jumlah_positif = Y.sum().sort_values(ascending=False)

    print("\nJumlah data untuk setiap label (nilai 1):")
    display(
        jumlah_positif.rename_axis("Label").reset_index(name="Jumlah Data")
    )

    # Hitung berapa label yang dimiliki setiap baris
    label_per_data = Y.sum(axis=1)
    distribusi_label = label_per_data.value_counts().sort_index()

    print("\nDistribusi jumlah label pada setiap data:")
    display(
        distribusi_label.rename_axis("Jumlah Label Aktif")
        .reset_index(name="Jumlah Data")
    )

    plt.figure(figsize=(9, 4))
    jumlah_positif.plot(kind="bar")
    plt.title(f"Jumlah Data Positif per Label - {nama_dataset}")
    plt.xlabel("Nama Label")
    plt.ylabel("Jumlah Data dengan Label = 1")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    data_multi_label = (label_per_data > 1).sum()
    print("\nData dengan lebih dari satu label aktif:", data_multi_label)

    if Y.shape[1] > 1 and data_multi_label > 0:
        print("KESIMPULAN: Dataset memenuhi karakteristik MULTILABEL.")
    else:
        print("PERHATIAN: Belum ditemukan data dengan lebih dari satu label aktif.")

    return X, Y

## 3. Dataset Pertama: Flags

**Sumber:** Cometa – Flags  
https://cometa.ujaen.es/datasets/flags

Dataset Flags berisi karakteristik bendera negara, seperti warna, bentuk, dan simbol. Label targetnya menunjukkan keberadaan warna tertentu pada bendera, misalnya merah, hijau, biru, kuning, putih, hitam, dan oranye.

Dataset ini memiliki 194 data, 19 fitur masukan, dan 7 label. Satu bendera dapat memiliki beberapa warna sekaligus, sehingga satu baris data dapat memiliki beberapa label bernilai `1`.


In [ ]:
X_flags, Y_flags = periksa_multilabel(
    nama_dataset="Flags",
    nama_file="flags.arff",
    jumlah_label=7,
    url_sumber="https://cometa.ujaen.es/datasets/flags"
)

### Alasan termasuk multilabel
Setiap data pada Flags mewakili satu bendera. Sebuah bendera dapat memiliki beberapa warna sekaligus. Karena itu, beberapa kolom label dapat bernilai `1` pada baris yang sama. Hal ini merupakan karakteristik klasifikasi multilabel.


## 4. Dataset Kedua: Emotions

**Sumber:** Cometa – Emotions  
https://cometa.ujaen.es/datasets/emotions

Dataset Emotions berisi fitur audio dari musik dan label emosi yang dapat ditimbulkan oleh musik tersebut. Enam labelnya adalah amazed-surprised, happy-pleased, relaxing-calm, quiet-still, sad-lonely, dan angry-aggressive.

Dataset ini memiliki 593 data, 72 fitur masukan, dan 6 label. Satu musik dapat dikaitkan dengan lebih dari satu emosi.


In [ ]:
X_emotions, Y_emotions = periksa_multilabel(
    nama_dataset="Emotions",
    nama_file="emotions.arff",
    jumlah_label=6,
    url_sumber="https://cometa.ujaen.es/datasets/emotions"
)

### Alasan termasuk multilabel
Satu musik tidak harus hanya memiliki satu emosi. Sebagai contoh, sebuah lagu dapat memiliki label happy-pleased sekaligus relaxing-calm. Karena satu data dapat memiliki beberapa label target secara bersamaan, Emotions termasuk klasifikasi multilabel.


## 5. Dataset Ketiga: Birds

**Sumber:** Cometa – Birds  
https://cometa.ujaen.es/datasets/birds

Dataset Birds berisi fitur audio untuk mengenali keberadaan berbagai spesies burung dalam rekaman suara. Dataset ini memiliki 645 data, 260 fitur masukan, dan 19 label spesies burung.

Satu rekaman suara dapat berisi suara beberapa spesies burung sekaligus. Oleh sebab itu, satu rekaman dapat memiliki lebih dari satu label.


In [ ]:
X_birds, Y_birds = periksa_multilabel(
    nama_dataset="Birds",
    nama_file="birds.arff",
    jumlah_label=19,
    url_sumber="https://cometa.ujaen.es/datasets/birds"
)

### Alasan termasuk multilabel
Dalam satu rekaman suara, bisa terdengar lebih dari satu spesies burung. Dengan demikian, satu baris data dapat mempunyai beberapa label spesies yang bernilai `1`. Karena targetnya dapat terdiri dari beberapa label sekaligus, Birds termasuk klasifikasi multilabel.


## 6. Kesimpulan

Berdasarkan pemeriksaan ketiga dataset:

1. **Flags** termasuk multilabel karena satu bendera dapat memiliki beberapa warna sekaligus.
2. **Emotions** termasuk multilabel karena satu musik dapat memiliki beberapa label emosi.
3. **Birds** termasuk multilabel karena satu rekaman suara dapat mengandung beberapa spesies burung.

Ketiga dataset memiliki beberapa kolom target biner. Satu baris data dapat mempunyai lebih dari satu label bernilai `1`. Hal tersebut membedakan multilabel dari binary dan multiclass.

**Catatan:** Jumlah data, fitur, label, dan distribusi yang ditampilkan merupakan hasil pembacaan file ARFF yang digunakan. Sumber dan deskripsi dataset dicantumkan pada masing-masing bagian.
